In [ ]:
%pip install VESIcal # you need this to run the saturation pressure calcs. 

In [ ]:
%pip install petthermotools # remember to comment these out once you are done

Note: you may need to restart the kernel to use updated packages.


In [29]:
# import core python packages that we'll use for plotting and data manipulation.
import sys
import os
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import seaborn as sns
import petthermotools as ptt 
import VESIcal as v

In [4]:
# Lets see what models we have - you wont be able to run magmasat. 
v.get_model_names()

['ShishkinaIdealMixing',
 'Dixon',
 'IaconoMarziano',
 'Liu',
 'ShishkinaCarbon',
 'ShishkinaWater',
 'DixonCarbon',
 'DixonWater',
 'IaconoMarzianoCarbon',
 'IaconoMarzianoWater',
 'AllisonCarbon',
 'AllisonCarbon_sunset',
 'AllisonCarbon_sfvf',
 'AllisonCarbon_erebus',
 'AllisonCarbon_vesuvius',
 'AllisonCarbon_etna',
 'AllisonCarbon_stromboli',
 'MooreWater',
 'LiuWater',
 'LiuCarbon']

## Q1 - lets gain some intuition as to how much volatiles we can dissolve in a given magma using different models

In [5]:
# Lets load the major elements for 1 sample - This is a melt inclusion from Kilauea
kilauea = v.Sample({'SiO2':  48.42,
                    'TiO2':   2.45,
                    'Al2O3': 11.90,
                    'Fe2O3':  0.00,
                    'FeO':   11.33,
                    'MgO':   12.51,
                    'CaO':   10.02,
                    'Na2O':   2.10,
                    'K2O':    0.45,
                    'P2O5':   0.30,
                    })

# Now lets calculate the amount of dissolved volatiles in a given magma using the Kilauea composition above, and the following conditions
# Pressure in bars, T in celcius, XFluid, 1 is all H2O, 0 is all CO2
calc_IM = v.calculate_dissolved_volatiles(sample=kilauea, 
                                       pressure=1000, 
                                       temperature=1200, 
                                       X_fluid=0.05,
                                      model='IaconoMarziano')

# Lets take a look at our calculation results (shown in wt%)
calc_IM.result

{'H2O_liq': 0.7540099313023552, 'CO2_liq': 0.056696649828960936}

In [6]:
0.05314192542591163/0.056696649828960936

0.9373027433936046

In [7]:
0.05314*10**4

531.4

In [8]:
calc_IM

### 1a - How much CO2 can be dissolved at 1000 bars, 1200C, in a pure CO2 system? Here, in a pure CO2 system, X_fluid = mol fraction of water in the fluid = 0. So you will need to change X_fluid in the v.calculate_dissolved_volatiles function

In [9]:
calc_IM_PureCO2 = v.calculate_dissolved_volatiles(sample=kilauea, 
                                       pressure=1000, 
                                       temperature=1200, 
                                       X_fluid=  0,
                                      model='IaconoMarziano')

# Lets take a look at our calculation results (shown in wt%)
calc_IM_PureCO2.result

{'H2O_liq': 0.0, 'CO2_liq': 0.05314192542591163}

### 1b – Now change X_fluid =0.05 (e.g. 5% H2O in the fluid phase), and rerun the calc. What is the fractional change in CO2 content (e.g. Pure CO2 /CO2 0.05). Is this what you would expect from Henry’s law?

### 1c – How much CO2 can be dissolved at 3000 bars, 1200C, in a pure CO2 system (so X_fluid = mol fraction of water in the fluid = 0)?

### 1d – Now rerun the calculations for X_fluid=0.05 at 3000 bars.  How does this fractional change in CO2 content at 3000 bars (e.g. Pure CO2 /CO2 0.05)  compare to the fractional change in CO2 content at 1000 bars?

### 1e – for 1000 and 3000 bars, do calculations at 10 different values of X_fluid. Save the values and plot an isobar diagram (H2O on x axis, CO2 on y axis), with your 10 datapoints. Your life is going to be easier if you write a forloop for this:

### 1f) How do you explain the larger change in CO2 for 3000 bars by changing XH2O=0.05 vs. 1000 bars?

### 1g) Now do the same calculation using model=’Dixon’, and overlay them on this plot. What are the differences? 

## Question 2- Lets look what happens as we decompress a magma 
- We will be using PetThermoTools, so run on VICTOR if you havent installed locally!

In [30]:
# Check the version
import petthermotools as ptt
ptt.__version__

'0.3.20'

In [31]:
## If using Mac run this cell **twice** 
import platform
if platform.system() == "Darwin" or platform.system() == "Linux":
    sys.stdout = open(os.devnull, 'w')
    sys.stderr = open(os.devnull, 'w')

In [32]:
# PetThermoTools has a slightly different input structure to VESIcal - input as a dictionary, not a custom data type
kilauea_primary = {'SiO2_Liq':  48.42,
                    'TiO2_Liq':   2.45,
                    'Al2O3_Liq': 11.90,
                    'Fe2O3_Liq':  2,
                    'FeO_Liq':   10,
                    'MgO_Liq':   12.51,
                    'CaO_Liq':   10.02,
                    'Na2O_Liq':   2.10,
                    'K2O_Liq':    0.45,
                    'P2O5_Liq':   0.30,
                    'H2O_Liq':0.5, 
                    'CO2_Liq': 5000/10**4
                    }

## Lets run a decompression calculation
### We are going to consider what happens when a primary Hawaiian melt ascends from 30 km depth (10,000 bars) up to 3 km (1000 bars). 
- Remember, Kil_decompress is a dictionary. To get the keys do Kil_decompress.keys(). The ‘All’ key has most things you need in it

In [33]:
Kil_decompress = ptt.isothermal_decompression(Model = "MELTSv1.2.0", # rhyolite-MELTS v1.2.0 - updated H2O-CO2 solubility model - This is 'Magmasat' from the lectures
                                            bulk = kilauea_primary, # composition defined above is our starting composition
                                            find_liquidus=True, # specify that we want to start at the liquidus 
                                            fO2_buffer="FMQ", # Impose a fO2 buffer
                                            P_start_bar=10000, # starting pressure (in bars)
                                            P_end_bar=1000, # final pressure (in bars)
                                            dp_bar=20) # pressure step (in bars)


A default timeout of 5 minutes has been specified. If you are not getting any results try increasing this using the timeout kwarg.
Running MELTSv1.2.0 calculation... Complete (time taken = 55.2 seconds)

In [34]:
Kil_decompress.keys()

dict_keys(['Conditions', 'liquid1', 'liquid1_prop', 'orthopyroxene1', 'orthopyroxene1_prop', 'fluid1', 'fluid1_prop', 'All', 'PhaseList', 'mass_g', 'volume_cm3', 'rho_kg/m3'])

In [35]:
# Lets take a look at 'All'
Kil_decompress['All']

,T_C,P_bar,mass_g,H_J,S_J/K,V_cm^3,rho_kg/m^3,log10(fO2),dVdP_cm^3/bar,SiO2_Liq,TiO2_Liq,Al2O3_Liq,Cr2O3_Liq,Fe2O3_Liq,FeO_Liq,FeOt_Liq,MnO_Liq,MgO_Liq,CaO_Liq,Na2O_Liq,K2O_Liq,P2O5_Liq,H2O_Liq,CO2_Liq,Fe3Fet_Liq,mass_g_Liq,rho_kg/m^3_Liq,V_cm^3_Liq,G_J_Liq,H_J_Liq,S_J/K_Liq,Cp_J/(kg.K^2)_Liq,dCpdT_J/(kg.K^2)_Liq,dVdT_cm^3/K_Liq,dPdT_bar/K_Liq,d2VdT2_cm^3/K^2_Liq,d2VdTdP_cm^3/(bar.K)_Liq,d2VdP2_cm^3/bar^2_Liq,molwt_Liq,SiO2_Opx,TiO2_Opx,Al2O3_Opx,Cr2O3_Opx,Fe2O3_Opx,FeO_Opx,FeOt_Opx,MnO_Opx,MgO_Opx,CaO_Opx,Na2O_Opx,K2O_Opx,P2O5_Opx,H2O_Opx,CO2_Opx,Fe3Fet_Opx,mass_g_Opx,rho_kg/m^3_Opx,V_cm^3_Opx,G_J_Opx,H_J_Opx,S_J/K_Opx,Cp_J/(kg.K^2)_Opx,dCpdT_J/(kg.K^2)_Opx,dVdT_cm^3/K_Opx,dPdT_bar/K_Opx,d2VdT2_cm^3/K^2_Opx,d2VdTdP_cm^3/(bar.K)_Opx,d2VdP2_cm^3/bar^2_Opx,molwt_Opx,H2O_Fl,CO2_Fl,X_H2O_mol_Fl,X_CO2_mol_Fl,mass_g_Fl,rho_kg/m^3_Fl,V_cm^3_Fl,G_J_Fl,H_J_Fl,S_J/K_Fl,Cp_J/(kg.K^2)_Fl,dCpdT_J/(kg.K^2)_Fl,dVdT_cm^3/K_Fl,dPdT_bar/K_Fl,d2VdT2_cm^3/K^2_Fl,d2VdTdP_cm^3/(bar.K)_Fl,d2VdP2_cm^3/bar^2_Fl,molwt_Fl
0,1407.170797,10000.0,99.967448,-1.133235e+06,290.044047,35.839712,2789.292719,-5.601401,-0.000165,47.884972,2.423168,11.769506,0.0,1.651780,10.182372,11.668632,0.0,12.370127,9.910218,2.077011,0.445074,0.296716,0.494527,0.494527,0.127372,99.958393,2789.261594,35.836866,-1.620449e+06,-1.133121e+06,290.020411,1532.700852,0.000174,0.003035,NaN,-2.807836e-08,-1.782411e-09,4.660259e-09,101.100088,55.630653,0.16441,2.659986,0.0,0.746549,7.106946,7.778687,0.0,32.484378,1.174632,0.032446,0.0,0.0,0.0,0.0,0.086356,0.009055,3181.160478,0.002846,-153.360507,-113.644516,0.023636,1302.526074,7.999214e-07,1.265388e-07,NaN,3.900314e-11,1.213331e-14,2.838808e-15,208.730344,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.000000,0.000000
1,1407.170797,9980.0,99.967470,-1.133293e+06,290.052556,35.843436,2789.003570,-5.602710,-0.000166,47.885664,2.422963,11.768678,0.0,1.651915,10.181895,11.668277,0.0,12.371947,9.909425,2.076826,0.445034,0.296689,0.494482,0.494482,0.127387,99.967470,2789.003570,35.843436,-1.620674e+06,-1.133293e+06,290.052556,1532.714442,0.000174,0.003036,NaN,-2.808217e-08,-1.781493e-09,4.660634e-09,101.100253,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.000000,0.000000
2,1407.170797,9960.0,99.967492,-1.133354e+06,290.058631,35.846766,2788.745049,-5.604020,-0.000166,47.885653,2.422963,11.768676,0.0,1.652136,10.181694,11.668275,0.0,12.371944,9.909423,2.076825,0.445034,0.296689,0.494482,0.494482,0.127404,99.967492,2788.745049,35.846766,-1.620746e+06,-1.133354e+06,290.058631,1532.715242,0.000174,0.003036,NaN,-2.808238e-08,-1.780572e-09,4.660626e-09,101.100133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.000000,0.000000
3,1407.170797,9940.0,99.967515,-1.133416e+06,290.064707,35.850099,2788.486430,-5.605329,-0.000166,47.885642,2.422962,11.768673,0.0,1.652357,10.181492,11.668272,0.0,12.371941,9.909420,2.076825,0.445034,0.296689,0.494482,0.494482,0.127421,99.967515,2788.486430,35.850099,-1.620818e+06,-1.133416e+06,290.064707,1532.716038,0.000174,0.003036,NaN,-2.808226e-08,-1.779633e-09,4.660614e-09,101.100014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,

### 2a- Plot ‘P_bar’ vs. ‘CO2_Liq’, and ‘P_bar’ vs. ‘H2O_Liq’, and screenshot below. Remember to label your axes. 

In [27]:
Kil_decompress['fluid1']

,H2O_Fl,CO2_Fl,X_H2O_mol_Fl,X_CO2_mol_Fl
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
...,...,...,...,...
446,2.405358,97.594642,0.056823,0.943177
447,2.444854,97.555146,0.057725,0.942275
448,2.485689,97.514311,0.058655,0.941345
449,2.527940,97.472060,0.059617,0.940383


### 2b) How much CO2 degasses vs. H2O? What would you speculate XH2O (proportion of H2O in the fluid phase would be)?


### 2c) Describe what happens to the system at ~6000 bars in terms of the physical phases present. 
(Hint the ['PhaseList'] key may be useful)

### 2e) Plot 'X_H2O_mol_Fl' against pressure. If this gas makes its way to the surface diffusively, what volatile phase do we expect to be dominant? 

## 3. Now lets let this magma start to fractionally crystallize after ascending into the main crustal magma storage reservoir

In [17]:
start_comp_after_ascent=Kil_decompress['liquid1'].iloc[-1] # this grabs the last value after the ascent calculation. 

In [18]:
Isobaric_Xtal = ptt.isobaric_crystallisation(Model = "MELTSv1.2.0",
                                           bulk = dict(start_comp_after_ascent),
                                           find_liquidus = True,
                                           P_bar = 1000,
                                           T_end_C = 1050,
                                           dt_C = 2,
                                           fO2_buffer = "FMQ",
                                           fO2_offset = +0.4,
                                           Frac_solid = True,
                                           Frac_fluid = True,
                                          )

A default timeout of 5 minutes has been specified. If you are not getting any results try increasing this using the timeout kwarg.
Running MELTSv1.2.0 calculation... Complete (time taken = 25.44 seconds)

In [19]:
Isobaric_Xtal.keys()

dict_keys(['Conditions', 'liquid1', 'liquid1_prop', 'fluid1', 'fluid1_prop', 'olivine1', 'olivine1_prop', 'clinopyroxene1', 'clinopyroxene1_prop', 'plagioclase1', 'plagioclase1_prop', 'spinel1', 'spinel1_prop', 'clinopyroxene2', 'clinopyroxene2_prop', 'orthopyroxene1', 'orthopyroxene1_prop', 'All', 'PhaseList', 'mass_g', 'volume_cm3', 'rho_kg/m3'])

### 3a – What is the order of phase appearance during crystallization? 
- (Hint, the ['PhaseList'] key might be useful here)
- you can use .to_clipboard(excel=True) to copy and paste into excel to easily scroll through what phases appear

### 3b – At what point does the magma saturate in volatiles? 

### 3c –We are currently running this as an open system for H2O and CO2, so the moment the bubbles form, they escape. Plot what happens to H2O and CO2 in the melt against the MgO content of the melt.

### 3d – Why do they do different things? Hint: plot the fluid composition against MgO and the cumulative amount of fluid exsolved against MgO (you can find this under the ‘mass_g’ key in the dictionary, the _cumsum columns show the cumulative sum, the other columns in show the amount of phase formed at each step). What happens to the amount of fluid during FC? 

## Lets try keeping the fluid around instead of letting it escape at each step. 
### 3e – Now run the calculation with Frac_fluid=False. This will keep the fluid in the system. Look at the ‘volume_cm3’ dataframe. Calculate the % of fluid, where % fluid = 100*fluid volume/(liquid volume+fluid vol). Plot this against MgO. The % fluid in the system is the amount of vesicularity – e.g. the vol% of the bubbles in the melt. 

In [20]:
Isobaric_Xtal_keepfluid = ptt.isobaric_crystallisation(Model = "MELTSv1.2.0",
                                           bulk = dict(start_comp_after_ascent),
                                           find_liquidus = True,
                                           P_bar = 1000,
                                           T_end_C = 1050,
                                           dt_C = 2,
                                           fO2_buffer = "FMQ",
                                           fO2_offset = +0.4,
                                           Frac_solid = True,
                                           Frac_fluid = False,
                                          )

A default timeout of 5 minutes has been specified. If you are not getting any results try increasing this using the timeout kwarg.
Running MELTSv1.2.0 calculation... Complete (time taken = 25.43 seconds)

In [21]:
Isobaric_Xtal_keepfluid.keys()

dict_keys(['Conditions', 'liquid1', 'liquid1_prop', 'fluid1', 'fluid1_prop', 'olivine1', 'olivine1_prop', 'clinopyroxene1', 'clinopyroxene1_prop', 'plagioclase1', 'plagioclase1_prop', 'spinel1', 'spinel1_prop', 'clinopyroxene2', 'clinopyroxene2_prop', 'orthopyroxene1', 'orthopyroxene1_prop', 'All', 'PhaseList', 'mass_g', 'volume_cm3', 'rho_kg/m3'])

In [22]:
Isobaric_Xtal_keepfluid['volume_cm3']

,liquid1,fluid1,olivine1,clinopyroxene1,plagioclase1,spinel1,clinopyroxene2,orthopyroxene1
0,37.026170,0.009874,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
1,36.943149,0.011757,0.071318,0.000000,0.000000,0.000000,0.0,0.000000
2,36.858322,0.013674,0.073471,0.000000,0.000000,0.000000,0.0,0.000000
3,36.774127,0.015570,0.072910,0.000000,0.000000,0.000000,0.0,0.000000
4,36.690556,0.017446,0.072354,0.000000,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...
117,12.631597,0.294380,0.000000,0.031072,0.091721,0.017534,0.0,0.014026
118,12.472491,0.297006,0.000000,0.029567,0.089383,0.016718,0.0,0.013540
119,12.318649,0.299597,0.000000,0.028169,0.087145,0.015956,0.0,0.013075
120,12.169788,0.302156,0.000000,0.026869,0.085001,0.015243,0.0,0.012628


#### Calculate the % of fluid, where % fluid = 100*fluid volume/(liquid volume+fluid vol). The % fluid in the system is the amount of vesicularity – e.g. the vol% of the bubbles in the melt. 

### Plot the % of fluid against against MgO.

## 4 – Lets get the magma to decompress up to the surface 
- (e.g. last step in the frac_fluid=True model) during an eruption. 
- we will use the model where we let fluids escape at every step so we dont have to deal with an exsolved volatile phase. 
-  This completes its journey from the mantle into the crust! 

In [23]:
comp_after_frac=Isobaric_Xtal['liquid1'].iloc[-1] # this grabs the last value after the crystallization calculation

In [24]:

Kil_decompress_closed= ptt.isothermal_decompression(Model = "MELTSv1.2.0", # rhyolite-MELTS v1.2.0 - updated H2O-CO2 solubility model - This is 'Magmasat' from the lectures
                                            bulk = comp_after_frac, # composition defined above is our starting composition
                                            find_liquidus=True, # specify that we want to start at the liquidus 
                                            fO2_buffer="FMQ", # Impose a fO2 buffer
                                            P_start_bar=1000, # starting pressure (in bars)
                                            P_end_bar=10, # final pressure (in bars)
                                            Frac_fluid=False,
                                            dp_bar=20) # pressure step (in bars)


A default timeout of 5 minutes has been specified. If you are not getting any results try increasing this using the timeout kwarg.
Running MELTSv1.2.0 calculation... Complete (time taken = 10.82 seconds)

In [25]:
Kil_decompress_closed.keys()

dict_keys(['Conditions', 'liquid1', 'liquid1_prop', 'fluid1', 'fluid1_prop', 'plagioclase1', 'plagioclase1_prop', 'orthopyroxene1', 'orthopyroxene1_prop', 'rhm-oxide1', 'rhm-oxide1_prop', 'clinopyroxene1', 'clinopyroxene1_prop', 'whitlockite1', 'whitlockite1_prop', 'All', 'PhaseList', 'mass_g', 'volume_cm3', 'rho_kg/m3'])

### 4a-What rock type is the initial composition? 
- Hint: You should know how to plot your data on a TAS Diagram from Homework 2, although you  will need to adjust the code


### 4b – Plot H2O (x axis) vs CO2 (y axis) in the liquid.  Also plot CO2 vs pressure and H2O vs pressure. Describe what happens to each volatile phase during degassing. 

### 4c – Plot Pressure (x axis) vs. SiO2 in the liquid. What happens to SiO2 during ascent? 

### 4e – Plot volume % volatiles vs. pressure. You can either reference volatiles to the liquid as above, or include the solid phases. Just state what you did. Describe what happens to vesicularity as you ascent